# 💧 AquaSense AI — 02: Preprocessing & Feature Engineering
**Project:** Intelligent Water Quality Assessment and Potability Prediction Using Explainable Machine Learning  

### Overview:
In this notebook, we demonstrate:
1. Domain-specific Feature Engineering (6 interaction terms).
2. Class-aware Grouped Median Imputation (without data leakage).
3. IQR Outlier Capping.
4. SMOTE Balancing on the training partition only.
5. Robust feature scaling.


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import DataLoader
from src.data.feature_engineer import WaterFeatureEngineer
from src.data.preprocessor import WaterQualityPreprocessor, GroupedMedianImputer, IQRCapper
from src.utils.visualization import set_plot_style
from src.utils.config import COLORS

set_plot_style()
print("Preprocessing modules loaded.")


## 1. Split Data (Stratified 80/20)


In [ ]:
dl = DataLoader(data_path="../data/raw/water_potability.csv")
df = dl.load()
X_train, X_test, y_train, y_test = dl.split(df, test_size=0.20, random_state=42)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")


## 2. Demonstrate Domain Feature Engineering
Construct domain interaction features:
- `ph_hardness_ratio`
- `tds_conductivity_ratio`
- `chloramine_organic_interaction`
- `trihalomethane_risk`
- `hardness_sulfate_ratio`
- `overall_contamination_index`


In [ ]:
fe = WaterFeatureEngineer()
X_train_fe = fe.fit_transform(X_train, y_train)
print(f"Features before FE: {X_train.shape[1]} -> Features after FE: {X_train_fe.shape[1]}")
X_train_fe[['ph_hardness_ratio', 'tds_conductivity_ratio', 'chloramine_organic_interaction', 'overall_contamination_index']].head()


## 3. Demonstrate Grouped Median Imputation
Imputes missing values using target group medians fitted on train partition.


In [ ]:
print("Missing values before imputation:")
print(X_train_fe[['ph', 'Sulfate', 'Trihalomethanes']].isnull().sum())

imputer = GroupedMedianImputer()
X_train_imp = imputer.fit_transform(X_train_fe, y_train)

print("
Missing values after imputation:")
print(X_train_imp[['ph', 'Sulfate', 'Trihalomethanes']].isnull().sum())


## 4. Demonstrate IQR Outlier Capping


In [ ]:
capper = IQRCapper(factor=1.5)
X_train_cap = capper.fit_transform(X_train_imp)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=X_train_imp['Solids'], ax=axes[0], color=COLORS['danger'])
axes[0].set_title("Solids Before Capping")

sns.boxplot(y=X_train_cap['Solids'], ax=axes[1], color=COLORS['safe'])
axes[1].set_title("Solids After 1.5x IQR Capping")
plt.tight_layout()
plt.show()


## 5. End-to-End Pipeline & SMOTE Balancing


In [ ]:
preprocessor = WaterQualityPreprocessor()
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc = preprocessor.transform(X_test)

print("Class counts before SMOTE:", y_train.value_counts().to_dict())
X_train_bal, y_train_bal = preprocessor.apply_smote(X_train_proc, y_train)
print("Class counts after SMOTE:", y_train_bal.value_counts().to_dict())

preprocessor.save("../models/preprocessor.pkl")
print("Fitted preprocessor serialized successfully.")
